In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe078.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
#path = '/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/211028_af02/ukb_wes_200k_af02_chr21_ptv_ko.vcf.bgz'
#mt = hl.import_vcf(path)

#ht = hl.import_table('/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/200k/UKBB_WES200k_filtered_binary_phenotypes.tsv.gz', force_bgz = True, impute = True)
#ht.describe()

input_phased_path = "data/mt/ukb_wes_200k_annotated_chr21.mt"
input_unphased_path = "data/mt/ukb_wes_200k_annotated_chr21_singletons.mt"

mt1 = qc.get_table(input_path=input_phased_path, input_type='mt') 
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt')

In [4]:
mt2.count()

(59374, 199795)

In [5]:
#mt1 = qc.filter_max_af(mt1, float(0.5))
#mt2 = qc.filter_max_af(mt2, float(0.5))

In [6]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
mt2 = mt2.explode_rows(mt2.consequence.vep.worst_csq_by_gene_canonical)
    
# get VEP annotation and add to rows
by_gene_annotation1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_by_gene_canonical, mt1.consequence.dbnsfp, use_loftee = False)
by_gene_annotation2 = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_by_gene_canonical, mt2.consequence.dbnsfp, use_loftee = False)
mt1 = mt1.annotate_rows(consequence_category = by_gene_annotation1)    
mt2 = mt2.annotate_rows(consequence_category = by_gene_annotation2) 

In [24]:
#mt = analysis.count_urv_by_samples(mt2)
#mt = analysis.count_urv_by_genes(mt)
#mt.describe()
#items = ['ptv','damaging_missense']
#mt1_subset = mt1.filter_rows(hl.literal(set(items)).contains(mt1.consequence_category)) 
#mt2_subset = mt2.filter_rows(hl.literal(set(items)).contains(mt2.consequence_category))

def count_urv_by_genes(mt):
    r''' Count up URVs by gene. Requires using the "count_urv_by_sample" function first.
    '''
    return (mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate(by_gene = hl.struct(
                URV_PTV = hl.agg.sum(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                URV_PTV_LC = hl.agg.sum(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                URV_non_coding = hl.agg.sum(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
            )      
        )
    )

test = count_urv_by_genes(mt2)
test = test.entries()
test.filter(test.by_gene.URV_PTV > 0).show()

2021-11-03 11:29:42 Hail: INFO: Ordering unsorted dataset with network shuffle


+-------------------+-----------+-------+-----------------+--------------------+------------------------+
| gene_id           | s         |   eur | by_gene.URV_PTV | by_gene.URV_PTV_LC | by_gene.URV_non_coding |
+-------------------+-----------+-------+-----------------+--------------------+------------------------+
| str               | str       | int32 |           int64 |              int64 |                  int64 |
+-------------------+-----------+-------+-----------------+--------------------+------------------------+
| "ENSG00000141956" | "1021606" |    NA |               1 |                  0 |                      2 |
| "ENSG00000141956" | "1170123" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "1264543" |    NA |               1 |                  0 |                      2 |
| "ENSG00000141956" | "1766243" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "1934996" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "1946172" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2001246" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2016178" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2101511" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2113741" |    NA |               1 |                  0 |                      1 |
| "ENSG00000141956" | "2113820" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2134978" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2448243" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2455014" |    NA |               1 |                  0 |                      1 |
| "ENSG00000141956" | "2515225" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2689651" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2697536" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2822768" |    NA |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2959959" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "3011861" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3122438" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3266388" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3374668" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3520405" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "3551898" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3560934" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3601131" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3670989" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "3710287" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3740496" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "4248321" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "4339423" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "4389384" |    NA |               1 |                  0

In [23]:
def count_urv_by_genes2(mt):
    r''' Count up URVs by gene. Requires using the "count_urv_by_sample" function first.
    '''
    return (mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate(by_gene = hl.struct(
                URV_PTV = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                URV_PTV_LC = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                URV_non_coding = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
            )      
        )
    )

test2 = count_urv_by_genes2(mt2)
test2 = test2.entries()
test2.filter(test2.by_gene.URV_PTV > 0).show()
#test2.show()

2021-11-03 11:24:23 Hail: INFO: Ordering unsorted dataset with network shuffle


+-------------------+-----------+-------+-----------------+--------------------+------------------------+
| gene_id           | s         |   eur | by_gene.URV_PTV | by_gene.URV_PTV_LC | by_gene.URV_non_coding |
+-------------------+-----------+-------+-----------------+--------------------+------------------------+
| str               | str       | int32 |           int64 |              int64 |                  int64 |
+-------------------+-----------+-------+-----------------+--------------------+------------------------+
| "ENSG00000141956" | "1021606" |    NA |               1 |                  0 |                      2 |
| "ENSG00000141956" | "1170123" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "1264543" |    NA |               1 |                  0 |                      2 |
| "ENSG00000141956" | "1766243" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "1934996" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "1946172" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2001246" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2016178" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2101511" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2113741" |    NA |               1 |                  0 |                      1 |
| "ENSG00000141956" | "2113820" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2134978" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2448243" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2455014" |    NA |               1 |                  0 |                      1 |
| "ENSG00000141956" | "2515225" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "2689651" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2697536" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2822768" |    NA |               1 |                  0 |                      2 |
| "ENSG00000141956" | "2959959" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "3011861" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3122438" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3266388" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3374668" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3520405" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "3551898" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3560934" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3601131" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3670989" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "3710287" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "3740496" |    NA |               1 |                  0 |                      0 |
| "ENSG00000141956" | "4248321" |     1 |               1 |                  0 |                      2 |
| "ENSG00000141956" | "4339423" |     1 |               1 |                  0 |                      0 |
| "ENSG00000141956" | "4389384" |    NA |               1 |                  0

In [30]:
def count_urv_by_genes(mt):
    r''' Count up URVs by gene (as defined by vep.worst_csq_for_variant_canonical.gene_id) '''
    return (mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate(gene = hl.struct(
                n_coding_URV_SNP = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                n_coding_URV_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                n_URV_PTV = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                n_URV_PTV_LC = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                n_URV_damaging_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "damaging_missense")),
                n_URV_other_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "other_missense")),
                n_URV_synonymous = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "synonymous")),
                n_URV_non_coding = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
            )      
        )
    )

test = count_urv_by_genes(mt2)
test.entries().flatten()





----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'gene_id': str 
    's': str 
    'eur': int32 
    'gene.n_coding_URV_SNP': int64 
    'gene.n_coding_URV_indel': int64 
    'gene.n_URV_PTV': int64 
    'gene.n_URV_PTV_LC': int64 
    'gene.n_URV_damaging_missense': int64 
    'gene.n_URV_other_missense': int64 
    'gene.n_URV_synonymous': int64 
    'gene.n_URV_non_coding': int64 
----------------------------------------
Key: []
----------------------------------------


2021-11-03 11:44:53 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-03 11:56:18 Hail: INFO: Ordering unsorted dataset with network shuffle


In [32]:
mt1_subset.count()

2021-10-29 11:41:51 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-10-29 11:42:01 Hail: INFO: Coerced sorted dataset


(3823, 185506)

In [33]:
mt2_subset.count()

(0, 199795)

In [38]:
mt = analysis.gene_csqs_calc_pKO_pseudoSNP(mt1_subset, mt2_subset, 21)

In [39]:
mt.count()

FatalError: FileNotFoundException: File data/mt/ukb_wes_200k_annotated_chr21.mt/rows/rows/metadata.json.gz does not exist

Java stack trace:
is.hail.utils.HailException: failed to read RVD spec data/mt/ukb_wes_200k_annotated_chr21.mt/rows/rows
	at is.hail.utils.ErrorHandling.fatal(ErrorHandling.scala:15)
	at is.hail.utils.ErrorHandling.fatal$(ErrorHandling.scala:15)
	at is.hail.utils.package$.fatal(package.scala:78)
	at is.hail.rvd.AbstractRVDSpec$.read(AbstractRVDSpec.scala:50)
	at is.hail.expr.ir.RVDComponentSpec.rvdSpec(AbstractMatrixTableSpec.scala:106)
	at is.hail.expr.ir.TableNativeZippedReader.leftPType(TableIR.scala:1071)
	at is.hail.expr.ir.TableNativeZippedReader.rowAndGlobalPTypes(TableIR.scala:1079)
	at is.hail.expr.ir.Requiredness.analyzeTable(Requiredness.scala:306)
	at is.hail.expr.ir.Requiredness.analyze(Requiredness.scala:292)
	at is.hail.expr.ir.Requiredness.run(Requiredness.scala:95)
	at is.hail.expr.ir.Requiredness$.apply(Requiredness.scala:18)
	at is.hail.expr.ir.R

In [11]:
 # iterate over the following consequence categories    
    categories = dict(
        ptv = ['ptv'],
        ptv_damaging_missense = ['ptv','damaging_missense'],
        synonymous = ['synonymous']
    )


2021-10-29 11:24:19 Hail: INFO: Coerced sorted dataset


(0, 185506)

In [64]:
#test1 = (ht['genetic.eur.no.fin.oct2021']).collect() 
#test2 = (ht['genetic.eur.oct2021']).collect()

In [82]:
def summary_count_urv(mt):
    annotation_by_variant = analysis.annotation_case_builder(mt.vep.worst_csq_for_variant_canonical, mt.dbnsfp)
    mt = mt.annotate_rows(consequence_category = annotation_by_variant)
    mt = analysis.count_urv_by_samples(mt)
    return mt.cols()

def summary_count_homozygous_urv(mt):
    annotation_by_variant = analysis.annotation_case_builder(mt.vep.worst_csq_for_variant_canonical, mt.dbnsfp)
    mt = mt.annotate_rows(consequence_category = annotation_by_variant)
    mt = analysis.count_homozygous_urv_by_samples(mt)
    return mt.cols()

In [99]:
mt1 = hl.read_matrix_table('data/mt_new/ukb_wes_200k_annotated_chr21.mt')
#mt2 = hl.read_matrix_table('data/mt_new/ukb_wes_200k_annotated_chr21_singletons.mt')

In [100]:
consequence_annotations = hl.read_table('data/vep/hail/ukb_wes_200k_chr21_vep.ht')
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 
#mt2 = mt2.annotate_rows(consequence = consequence_annotations[mt2.row_key]) 

In [85]:
#mt1 = mt1.explode_rows(mt1.vep.worst_csq_by_gene_canonical)
#mt2 = mt2.explode_rows(mt2.vep.worst_csq_by_gene_canonical)

In [86]:
#by_gene_annotation1 = analysis.annotation_case_builder(mt1.vep.worst_csq_by_gene_canonical, mt1.dbnsfp, use_loftee = False)
#by_gene_annotation2 = analysis.annotation_case_builder(mt2.vep.worst_csq_by_gene_canonical, mt2.dbnsfp, use_loftee = False)
#mt1 = mt1.annotate_rows(consequence_category = by_gene_annotation1)    
#mt2 = mt2.annotate_rows(consequence_category = by_gene_annotation2) 

In [92]:
summary_count_homozygous_urv(mt2).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    's': str 
    'eur': int32 
    'n_hom_coding_URV_SNP': int64 
    'n_hom_coding_URV_indel': int64 
    'n_hom_URV_PTV': int64 
    'n_hom_URV_PTV_LC': int64 
    'n_hom_URV_damaging_missense': int64 
    'n_hom_URV_other_missense': int64 
    'n_hom_URV_synonymous': int64 
    'n_hom_URV_non_coding': int64 
----------------------------------------
Key: ['s']
----------------------------------------


In [93]:
path = '/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/data/qc_final/ukb_wes_200k_chr21_variants_phased.ht'

In [98]:
ht = hl.read_matrix_table(path)

#mt = hl.read_matrix_table('data/mt/ukb_wes_200k_anotated_chr21.mt')
#consequence_annotations = hl.read_table('data/vep/hail/ukb_wes_200k_chr21_vep.ht')

+----------------+-------------------------------+---------------+----------------+---------------+-----------------------------+----------------------+---------------------+-------------------------+
| locus          | alleles                       | variant_qc.AC | variant_qc.AF  | variant_qc.AN | variant_qc.homozygote_count | variant_qc.call_rate | variant_qc.n_called | variant_qc.n_not_called |
+----------------+-------------------------------+---------------+----------------+---------------+-----------------------------+----------------------+---------------------+-------------------------+
| locus<GRCh38>  | array<str>                    | array<int32>  | array<float64> |         int32 | array<int32>                |              float64 |               int64 |                   int64 |
+----------------+-------------------------------+---------------+----------------+---------------+-----------------------------+----------------------+---------------------+-------------------------+
| chr21:10413617 | ["C","T"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413618 | ["G","A"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413629 | ["C","T"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413634 | ["T","G"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413636 | ["G","T"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413653 | ["A","G"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413662 | ["C","T"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413708 | ["TGGAGCAGTAAGATGGCGGCC","T"] | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413748 | ["T","C"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413764 | ["C","T"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413783 | ["A","G"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413787 | ["C","T"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413790 | ["G","C"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413791 | ["A","AG"]                    | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |                       0 |
| chr21:10413798 | ["G","A"]                     | [0,0]         | NA             |             0 | [0,0]                       |                  NaN |                   0 |       

In [19]:
mt = mt.annotate_rows(consequence = consequence_annotations[mt.row_key]) 

In [21]:
mt.vep.worst_csq_by_gene_canonical

<StructExpression of type struct{allele_num: int32, amino_acids: str, biotype: str, canonical: int32, ccds: str, cdna_start: int32, cdna_end: int32, cds_end: int32, cds_start: int32, codons: str, consequence_terms: array<str>, distance: int32, domains: array<struct{db: str, name: str}>, exon: str, gene_id: str, gene_pheno: int32, gene_symbol: str, gene_symbol_source: str, hgnc_id: str, hgvsc: str, hgvsp: str, hgvs_offset: int32, impact: str, intron: str, lof: str, lof_flags: str, lof_filter: str, lof_info: str, minimised: int32, polyphen_prediction: str, polyphen_score: float64, protein_end: int32, protein_start: int32, protein_id: str, sift_prediction: str, sift_score: float64, strand: int32, swissprot: str, transcript_id: str, trembl: str, uniparc: str, variant_allele: str, most_severe_consequence: str, csq_score: float64}>

In [15]:
categories = dict(
    ptv = ['ptv'],
    ptv_damaging_missense = ['ptv','damaging_missense'],
    synonymous = ['synonymous']
)
categories

{'ptv': ['ptv'],
 'ptv_damaging_missense': ['ptv', 'damaging_missense'],
 'synonymous': ['synonymous']}

In [17]:
for category, items in categories.items():
    print(category)
    print(items)

ptv
['ptv']
ptv_damaging_missense
['ptv', 'damaging_missense']
synonymous
['synonymous']


In [8]:
mt = hl.read_matrix_table('data/mt/ukb_wes_200k_annotated_chr21.mt')
annotation_by_variant = analysis.annotation_case_builder(mt.vep.worst_csq_for_variant_canonical, mt.dbnsfp)
annotation_by_gene = analysis.annotation_case_builder(mt.vep.worst_csq_by_gene_canonical, mt.dbnsfp)
mt = mt.annotate_rows(consequence_category = annotation_by_variant)
mt = analysis.count_homozygous_urv_by_samples(mt)
mt.cols().export('test.tsv.bgz')

2021-10-19 18:48:58 Hail: INFO: Coerced sorted dataset
2021-10-19 18:48:58 Hail: INFO: merging 16 files totalling 1.6M...
2021-10-19 18:48:58 Hail: INFO: while writing:
    test.tsv.bgz
  merge time: 20.812ms


In [9]:
def summary_count_urv(mt):
    annotation_by_variant = analysis.annotation_case_builder(mt.vep.worst_csq_for_variant_canonical, mt.dbnsfp)
    mt = mt.annotate_rows(consequence_category = annotation_by_variant)
    mt = analysis.counts_urv_by_samples(mt)
    return mt.cols()

def summary_count_homozygoys_urv(mt):
    annotation_by_variant = analysis.annotation_case_builder(mt.vep.worst_csq_for_variant_canonical, mt.dbnsfp)
    mt = mt.annotate_rows(consequence_category = annotation_by_variant)
    mt = analysis.count_homozygous_urv_by_samples(mt)
    return mt.cols()

In [6]:
def count_homozygous_urv_by_samples(mt):
    r'''Count up homozygous variants by samples
    
    :param mt: a MatrixTable with the field "consequence_category"
    '''
    return mt.annotate_cols(n_hom_coding_URV_SNP = hl.agg.count_where(mt.GT.is_hom_var() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_hom_coding_URV_indel = hl.agg.count_where(mt.GT.is_hom_var() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_hom_URV_PTV = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "ptv")),
                          n_hom_URV_PTV_LC = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "ptv_lc")),
                          n_hom_URV_damaging_missense = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "damaging_missense")),
                          n_hom_URV_other_missense = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "other_missense")),
                          n_hom_URV_synonymous = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "synonymous")),
                          n_hom_URV_non_coding = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "non_coding"))
                         )

In [92]:
def count_urv_by_samples(mt):
    r'''Count up ultra rare variants by cols and cateogry
    
    :param mt: a MatrixTable with the field "consequence_category"
    '''
    return mt.annotate_cols(n_coding_URV_SNP = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_coding_URV_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_URV_PTV = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                          n_URV_PTV_LC = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                          n_URV_damaging_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "damaging_missense")),
                          n_URV_other_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "other_missense")),
                          n_URV_synonymous = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "synonymous")),
                          n_URV_non_coding = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
                         )

In [102]:
# get prefixes to out files
prefix = "/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/all/211013_ptv"
burden = prefix + "/ukb_wes_200k_af02_chr20_burden.tsv.gz"
ko_prob = prefix + "/ukb_wes_200k_af02_chr14_ko_prob.tsv.gz"
kos = prefix + "/ukb_wes_200k_af02_chr15_knockouts.tsv.gz"

#mt = annotate_urv_cols(mt)
#mt = mt.annotate_rows(AAF_bin = analysis.maf_category_case_builder(mt.info))

hl.import_table(kos)

FatalError: HailException: Cannot load file 'file:/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/all/211013_ptv/ukb_wes_200k_af02_chr15_knockouts.tsv.gz'
  .gz cannot be loaded in parallel. Is the file actually *block* gzipped?
  If the file is actually block gzipped (even though its extension is .gz),
  use the 'force_bgz' argument to treat all .gz file extensions as .bgz.
  If you are sure that you want to load a non-block-gzipped file serially
  on one core, use the 'force' argument.

Java stack trace:
is.hail.utils.HailException: Cannot load file 'file:/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/derived/knockouts/all/211013_ptv/ukb_wes_200k_af02_chr15_knockouts.tsv.gz'
  .gz cannot be loaded in parallel. Is the file actually *block* gzipped?
  If the file is actually block gzipped (even though its extension is .gz),
  use the 'force_bgz' argument to treat all .gz file extensions as .bgz.
  If you are sure that you want to load a non-block-gz

2021-10-19 15:23:24 Hail: INFO: Coerced sorted dataset


In [33]:
mt.group_rows_by(mt.AAF_bin).aggregate(hl.agg.sum(mt.n_coding_URV_SNP))

TypeError: aggregate() takes 1 positional argument but 2 were given

2021-10-19 11:58:30 Hail: INFO: Coerced sorted dataset


In [64]:
PLOF_CSQS = ["transcript_ablation", "splice_acceptor_variant",
             "splice_donor_variant", "stop_gained", "frameshift_variant"]

MISSENSE_CSQS = ["stop_lost", "start_lost", "transcript_amplification",
                 "inframe_insertion", "inframe_deletion", "missense_variant"]

SYNONYMOUS_CSQS = ["stop_retained_variant", "synonymous_variant"]

OTHER_CSQS = ["mature_miRNA_variant", "5_prime_UTR_variant",
              "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant",
              "NMD_transcript_variant", "non_coding_transcript_variant", "upstream_gene_variant",
              "downstream_gene_variant", "TFBS_ablation", "TFBS_amplification", "TF_binding_site_variant",
              "regulatory_region_ablation", "regulatory_region_amplification", "feature_elongation",
              "regulatory_region_variant", "feature_truncation", "intergenic_variant"]


In [63]:
def variant_csqs_category_builder(csqs_vep_expr, csqs_dbnsfp_expr):
    r'''Create categories for downstream analysis'''
    return (hl.case()
            .when(hl.literal(set(PLOF_CSQS)).contains(csqs_stats_expr), "ptv")
            .when(hl.literal(set(MISSENSE_CSQS)).contains(csqs_stats_expr) & 
                   (~hl.is_defined(csqs_dbnsfp_expr.cadd_phred_score) | 
                    ~hl.is_defined(csqs_dbnsfp_expr.revel_score)), "other_missense")                                   
                .when(hl.literal(set(MISSENSE_CSQS)).contains(mt.vep.worst_csq_by_gene_canonical.most_severe_consequence) & 
                   (csqs_dbnsfp_expr.cadd_phred_score >= 20) & 
                   (csqs_dbnsfp_expr.revel_score >= 0.6), "damaging_missense") 
                .when(hl.literal(set(MISSENSE_CSQS)).contains(csqs_vep_expr), "other_missense")
                .when(hl.literal(set(SYNONYMOUS_CSQS)).contains(csqs_vep_expr), "synonymous")
                .when(hl.literal(set(OTHER_CSQS)).contains(csqs_vep_expr), "non_coding")
                .default("NA"))

In [78]:
def annotation_case_builder(worst_csq_by_gene_canonical_expr, csqs_dbnsfp_expr, use_loftee: bool = True):
    r'''Create categories for downstream analysis'''
    case = hl.case(missing_false=True)
    if use_loftee:
        case = (case
             .when(worst_csq_by_gene_canonical_expr.lof == 'HC', 'ptv')
             .when(worst_csq_by_gene_canonical_expr.lof == 'LC', 'ptv_LC')
            )
    else:
        case = case.when(hl.set(PLOF_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "ptv")
    case = (case.when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence) & 
                   (~hl.is_defined(csqs_dbnsfp_expr.cadd_phred_score) | ~hl.is_defined(csqs_dbnsfp_expr.revel_score)), "other_missense")                                   
                .when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence) & 
                   (csqs_dbnsfp_expr.cadd_phred_score >= 20) &  (csqs_dbnsfp_expr.revel_score >= 0.6), "damaging_missense") 
                .when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "other_missense")
                .when(hl.set(SYNONYMOUS_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "synonymous")
                .when(hl.set(OTHER_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), "non_coding")
           )
    return case.or_missing()
                
    

In [73]:
def annotation_case_builder(worst_csq_by_gene_canonical_expr, use_loftee: bool = True, use_polyphen_and_sift: bool = False,
                            strict_definitions: bool = False):
    case = hl.case(missing_false=True)
    if use_loftee:
        case = (case
                .when(worst_csq_by_gene_canonical_expr.lof == 'HC', 'pLoF')
                .when(worst_csq_by_gene_canonical_expr.lof == 'LC', 'LC'))
    else:
        case = case.when(hl.set(PLOF_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'pLoF')
    if use_polyphen_and_sift:
        case = (case
                .when(missense.contains(mt.vep.worst_csq_for_variant_canonical.most_severe_consequence) &
                      (mt.vep.worst_csq_for_variant_canonical.polyphen_prediction == "probably_damaging") &
                      (mt.vep.worst_csq_for_variant_canonical.sift_prediction == "deleterious"), "damaging_missense")
                .when(missense.contains(mt.vep.worst_csq_for_variant_canonical.most_severe_consequence), "other_missense"))
    else:
        if strict_definitions:
            case = case.when(worst_csq_by_gene_canonical_expr.most_severe_consequence == 'missense_variant', 'missense')
        else:
            case = case.when(hl.set(MISSENSE_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'missense')
    if strict_definitions:
        case = case.when(worst_csq_by_gene_canonical_expr.most_severe_consequence == 'synonymous_variant', 'synonymous')
    else:
        case = case.when(hl.set(SYNONYMOUS_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'synonymous')
    case = case.when(hl.set(OTHER_CSQS).contains(worst_csq_by_gene_canonical_expr.most_severe_consequence), 'non-coding')
    return case.or_missing()